# Solve

In [6]:
import numpy as np
import pandas as pd
import json

In [59]:
VELOCIDAD_MEDIA = 20/60/60 # 20 km/h en km/s

## Funciones

In [61]:
import math
def haversine(lat1, lon1, lat2, lon2):

      R = 3959.87433 # this is in miles.  For Earth radius in kilometers use 6372.8 km

      dLat = math.radians(lat2 - lat1)
      dLon = math.radians(lon2 - lon1)
      lat1 = math.radians(lat1)
      lat2 = math.radians(lat2)

      a = math.sin(dLat/2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dLon/2)**2
      c = 2*math.asin(math.sqrt(a))
      return R * c


## Instancia Small

### Preprocesado

In [89]:
# leer instancia 
data = json.load(open('instancias/medium.json'))
data.keys()

dict_keys(['viajes', 'vehiculos'])

In [90]:
data_viajes = pd.DataFrame(data['viajes'])
data_vehiculos = pd.DataFrame(data['vehiculos'])
# data_viajes, data_vehiculos

In [91]:
data_vehiculos['hora_inicio_jornada'] = data_vehiculos['inicio_jornada'].apply(lambda x: x['hora'])
data_vehiculos['lat_inicio_jornada'] = data_vehiculos['inicio_jornada'].apply(lambda x: x['lat'])
data_vehiculos['lon_inicio_jornada'] = data_vehiculos['inicio_jornada'].apply(lambda x: x['lon'])

data_vehiculos['hora_fin_jornada'] = data_vehiculos['fin_jornada'].apply(lambda x: x['hora'])
data_vehiculos['lat_fin_jornada'] = data_vehiculos['fin_jornada'].apply(lambda x: x['lat'])
data_vehiculos['lon_fin_jornada'] = data_vehiculos['fin_jornada'].apply(lambda x: x['lon'])

data_vehiculos.drop(columns=['inicio_jornada', 'fin_jornada'], inplace=True)
data_vehiculos

,id,capacidad,hora_inicio_jornada,lat_inicio_jornada,lon_inicio_jornada,hora_fin_jornada,lat_fin_jornada,lon_fin_jornada
0,VEH001,6,28800,-33.4489,-70.6693,54000,-33.4489,-70.6693
1,VEH002,6,28800,-33.4372,-70.6506,54000,-33.4372,-70.6506
2,VEH003,5,28800,-33.4258,-70.6106,54000,-33.4258,-70.6106
3,VEH004,6,28800,-33.4150,-70.6050,54000,-33.4150,-70.6050
4,VEH005,5,28800,-33.4320,-70.6180,54000,-33.4320,-70.6180


In [92]:
data_viajes

,id,origen,destino,hora_presentacion,num_pasajeros
0,V001,"{'lat': -33.4489, 'lon': -70.6693}","{'lat': -33.4372, 'lon': -70.6506}",28800,2
1,V002,"{'lat': -33.4258, 'lon': -70.6106}","{'lat': -33.4569, 'lon': -70.6483}",29100,3
2,V003,"{'lat': -33.415, 'lon': -70.605}","{'lat': -33.4489, 'lon': -70.6693}",29700,4
3,V004,"{'lat': -33.432, 'lon': -70.618}","{'lat': -33.445, 'lon': -70.66}",30300,1
4,V005,"{'lat': -33.45, 'lon': -70.67}","{'lat': -33.428, 'lon': -70.615}",30900,5
5,V006,"{'lat': -33.44, 'lon': -70.654}","{'lat': -33.42, 'lon': -70.61}",31500,2
6,V007,"{'lat': -33.41, 'lon': -70.6}","{'lat': -33.455, 'lon': -70.675}",32100,3
7,V008,"{'lat': -33.438, 'lon': -70.652}","{'lat': -33.418, 'lon': -70.608}",32700,4
8,V009,"{'lat': -33.447, 'lon': -70.665}","{'lat': -33.43, 'lon': -70.62}",33300,2
9,V010,"{'lat': -33.422, 'lon': -70.613}","{'lat': -33.451, 'lon': -70.668}",33900,3


In [93]:
data_viajes['lat_origen'] = data_viajes['origen'].apply(lambda x: x['lat'])
data_viajes['lon_origen'] = data_viajes['origen'].apply(lambda x: x['lon'])

data_viajes['lat_destino'] = data_viajes['destino'].apply(lambda x: x['lat'])
data_viajes['lon_destino'] = data_viajes['destino'].apply(lambda x: x['lon'])

data_viajes.drop(columns=['origen', 'destino'], inplace=True)
data_viajes

,id,hora_presentacion,num_pasajeros,lat_origen,lon_origen,lat_destino,lon_destino
0,V001,28800,2,-33.4489,-70.6693,-33.4372,-70.6506
1,V002,29100,3,-33.4258,-70.6106,-33.4569,-70.6483
2,V003,29700,4,-33.4150,-70.6050,-33.4489,-70.6693
3,V004,30300,1,-33.4320,-70.6180,-33.4450,-70.6600
4,V005,30900,5,-33.4500,-70.6700,-33.4280,-70.6150
5,V006,31500,2,-33.4400,-70.6540,-33.4200,-70.6100
6,V007,32100,3,-33.4100,-70.6000,-33.4550,-70.6750
7,V008,32700,4,-33.4380,-70.6520,-33.4180,-70.6080
8,V009,33300,2,-33.4470,-70.6650,-33.4300,-70.6200
9,V010,33900,3,-33.4220,-70.6130,-33.4510,-70.6680


In [94]:
data_viajes['tiempo_en_ruta'] = data_viajes.apply(lambda x: haversine(x['lat_origen'], x['lon_origen'], x['lat_destino'], x['lon_destino']) / VELOCIDAD_MEDIA, axis=1)
data_viajes['hora_finalizacion'] = data_viajes['hora_presentacion'] + data_viajes['tiempo_en_ruta']

### Prototipo

In [95]:
# def heuristica_solucion(data_viajes, data_vehiculos):
ids_veh = data_vehiculos['id']
# early_veh = {row['id']: row['hora_inicio_jornada'] for _, row in data_vehiculos.iterrows()}
# late_veh = {row['id']: row['hora_fin_jornada'] for _, row in data_vehiculos.iterrows()}
cap_veh = {row['id']: row['capacidad'] for _, row in data_vehiculos.iterrows()}
# Inicializo hora libre de cada vehículo con su hora de inicio de jornada
hora_libre = {vehiculo.id: vehiculo.hora_inicio_jornada for vehiculo in data_vehiculos.itertuples()}
ubicacion_libre = {row['id']: (row['lat_inicio_jornada'], row['lon_inicio_jornada']) for _, row in data_vehiculos.iterrows()}

sorted_viajes = data_viajes.sort_values(by='hora_presentacion')


In [96]:
viajes_no_asignados = []
rutas = {vehiculo.id: [] for vehiculo in data_vehiculos.itertuples()}
for viaje in sorted_viajes.itertuples():
    sin_asignar = True
    
    # Revisar si hay algún vehículo disponible en la ubicación del viaje a la hora de presentación

    # Asignar el viaje al vehículo más temprano disponible
    candidatos = []
    for vehiculo in data_vehiculos.itertuples():

        # Checkeo -1: hora de presentación mayor a inicio de ventana operativa
        if not(vehiculo.hora_inicio_jornada <= viaje.hora_presentacion):
            print("Hora de presentacion previa al inicio de la jornada")
            continue
        # Checkeo 0: que el vehículo tenga capacidad para la cantidad de pasajeros del viaje
        if not(cap_veh[vehiculo.id] >= viaje.num_pasajeros):
            print(f"El vehículo {vehiculo.id} no tiene capacidad para el viaje {viaje.id} \n{vehiculo.capacidad} < {viaje.num_pasajeros}")
            continue
            
        # Checkeo 1: que llegue a la hora de presentacion
        dh = haversine(ubicacion_libre[vehiculo.id][0], ubicacion_libre[vehiculo.id][1], viaje.lat_origen, viaje.lon_origen)
        tiempo_llegada = dh / VELOCIDAD_MEDIA
        hora_minima_llegada = hora_libre[vehiculo.id] + tiempo_llegada #type:ignore
        if not(hora_minima_llegada <= viaje.hora_presentacion): # type:ignore
            print(f"El vehículo {vehiculo.id} no llega a la hora de presentacion de {viaje.id}: {hora_libre[vehiculo.id]} + {tiempo_llegada} > {viaje.hora_presentacion}")
            continue
        
        # Checkeo 2: que el viaje + su retorno termine antes de la hora de fin de jornada del vehículo    
        hl = viaje.hora_presentacion + viaje.tiempo_en_ruta # type:ignore
        dh_retorno = haversine(viaje.lat_destino, viaje.lon_destino, vehiculo.lat_fin_jornada, vehiculo.lon_fin_jornada) 
        tiempo_viaje = dh_retorno / VELOCIDAD_MEDIA
        if not(hora_libre[vehiculo.id] + tiempo_llegada + tiempo_viaje <= vehiculo.hora_fin_jornada): # type:ignore
            continue
        
        espera = viaje.hora_presentacion - hora_minima_llegada
        candidatos.append((vehiculo.id, espera))
    print(viaje.id, candidatos)
    sorted_candidatos = sorted(candidatos, key=lambda x: x[1])
    vehiculo_asignado = sorted_candidatos[0][0] if len(sorted_candidatos) > 0 else None
    print(f"Viaje {viaje.id} asignado a vehículo {vehiculo_asignado}")

    # Actualizar hora libre y ubicación libre del vehículo asignado
    if vehiculo_asignado is not None:
        hora_libre[vehiculo_asignado] = viaje.hora_finalizacion # type:ignore
        ubicacion_libre[vehiculo_asignado] = (viaje.lat_destino, viaje.lon_destino)
        rutas[vehiculo_asignado].append(viaje.id)
    else:
        print(f"Viaje {viaje.id} no pudo ser asignado a ningún vehículo.")
        viajes_no_asignados.append(viaje.id)
    
print()
print("Viajes no asignados:", viajes_no_asignados)
print("Rutas asignadas a vehículos:", rutas)



El vehículo VEH002 no llega a la hora de presentacion de V001: 28800 + 242.62506097776435 > 28800
El vehículo VEH003 no llega a la hora de presentacion de V001: 28800 + 673.7426099949986 > 28800
El vehículo VEH004 no llega a la hora de presentacion de V001: 28800 + 789.6129379185406 > 28800
El vehículo VEH005 no llega a la hora de presentacion de V001: 28800 + 572.5400359260938 > 28800
V001 [('VEH001', 0.0)]
Viaje V001 asignado a vehículo VEH001
El vehículo VEH001 no llega a la hora de presentacion de V002: 29042.625060977763 + 438.82833757408116 > 29100
El vehículo VEH002 no llega a la hora de presentacion de V002: 28800 + 438.82833757408116 > 29100
V002 [('VEH003', 300.0), ('VEH004', 153.60192051346166), ('VEH005', 191.13430598449122)]
Viaje V002 asignado a vehículo VEH004
El vehículo VEH004 no llega a la hora de presentacion de V003: 29650.315411900578 + 688.3070970648585 > 29700
V003 [('VEH001', 109.2633891334408), ('VEH002', 351.8884501112043), ('VEH003', 753.6019205134617), ('VEH

## Heurística

In [99]:
def heuristica_solucion(data_viajes, data_vehiculos, verbose = False):
    ids_veh = data_vehiculos['id']
    # early_veh = {row['id']: row['hora_inicio_jornada'] for _, row in data_vehiculos.iterrows()}
    # late_veh = {row['id']: row['hora_fin_jornada'] for _, row in data_vehiculos.iterrows()}
    cap_veh = {row['id']: row['capacidad'] for _, row in data_vehiculos.iterrows()}
    # Inicializo hora libre de cada vehículo con su hora de inicio de jornada
    hora_libre = {vehiculo.id: vehiculo.hora_inicio_jornada for vehiculo in data_vehiculos.itertuples()}
    ubicacion_libre = {row['id']: (row['lat_inicio_jornada'], row['lon_inicio_jornada']) for _, row in data_vehiculos.iterrows()}

    sorted_viajes = data_viajes.sort_values(by='hora_presentacion')

    viajes_no_asignados = []
    rutas = {vehiculo.id: [] for vehiculo in data_vehiculos.itertuples()}
    for viaje in sorted_viajes.itertuples():
        sin_asignar = True
        
        # Revisar si hay algún vehículo disponible en la ubicación del viaje a la hora de presentación

        # Asignar el viaje al vehículo más temprano disponible
        candidatos = []
        for vehiculo in data_vehiculos.itertuples():

            # Checkeo -1: hora de presentación mayor a inicio de ventana operativa
            if not(vehiculo.hora_inicio_jornada <= viaje.hora_presentacion):
                if verbose: print("Hora de presentacion previa al inicio de la jornada")
                continue
            # Checkeo 0: que el vehículo tenga capacidad para la cantidad de pasajeros del viaje
            if not(cap_veh[vehiculo.id] >= viaje.num_pasajeros):
                if verbose: print(f"El vehículo {vehiculo.id} no tiene capacidad para el viaje {viaje.id} \n{vehiculo.capacidad} < {viaje.num_pasajeros}")
                continue
                
            # Checkeo 1: que llegue a la hora de presentacion
            dh = haversine(ubicacion_libre[vehiculo.id][0], ubicacion_libre[vehiculo.id][1], viaje.lat_origen, viaje.lon_origen)
            tiempo_llegada = dh / VELOCIDAD_MEDIA
            hora_minima_llegada = hora_libre[vehiculo.id] + tiempo_llegada #type:ignore
            if not(hora_minima_llegada <= viaje.hora_presentacion): # type:ignore
                if verbose: print(f"El vehículo {vehiculo.id} no llega a la hora de presentacion de {viaje.id}: {hora_libre[vehiculo.id]} + {tiempo_llegada} > {viaje.hora_presentacion}")
                continue
            
            # Checkeo 2: que el viaje + su retorno termine antes de la hora de fin de jornada del vehículo    
            hl = viaje.hora_presentacion + viaje.tiempo_en_ruta # type:ignore
            dh_retorno = haversine(viaje.lat_destino, viaje.lon_destino, vehiculo.lat_fin_jornada, vehiculo.lon_fin_jornada) 
            tiempo_viaje = dh_retorno / VELOCIDAD_MEDIA
            if not(hora_libre[vehiculo.id] + tiempo_llegada + tiempo_viaje <= vehiculo.hora_fin_jornada): # type:ignore
                continue
            
            espera = viaje.hora_presentacion - hora_minima_llegada
            candidatos.append((vehiculo.id, espera))
        if verbose: print(viaje.id, candidatos)
        sorted_candidatos = sorted(candidatos, key=lambda x: x[1])
        vehiculo_asignado = sorted_candidatos[0][0] if len(sorted_candidatos) > 0 else None
        if verbose: print(f"Viaje {viaje.id} asignado a vehículo {vehiculo_asignado}")

        # Actualizar hora libre y ubicación libre del vehículo asignado
        if vehiculo_asignado is not None:
            hora_libre[vehiculo_asignado] = viaje.hora_finalizacion # type:ignore
            ubicacion_libre[vehiculo_asignado] = (viaje.lat_destino, viaje.lon_destino)
            rutas[vehiculo_asignado].append(viaje.id)
        else:
            if verbose: print(f"Viaje {viaje.id} no pudo ser asignado a ningún vehículo.")
            viajes_no_asignados.append(viaje.id)
        
    if verbose: print()
    print("Viajes no asignados:", viajes_no_asignados)
    print("Rutas asignadas a vehículos:", rutas)
    return rutas, viajes_no_asignados


### Resolver

In [103]:
rutas_generadas, viajes_no_asignados = heuristica_solucion(data_viajes, data_vehiculos, verbose=False)

Viajes no asignados: []
Rutas asignadas a vehículos: {'VEH001': ['V001', 'V003', 'V006', 'V008', 'V010', 'V013', 'V020', 'V022', 'V023', 'V025', 'V027', 'V029'], 'VEH002': [], 'VEH003': ['V014', 'V016', 'V018', 'V021', 'V024', 'V026', 'V028', 'V030'], 'VEH004': ['V002', 'V004', 'V005', 'V007', 'V009', 'V011', 'V012', 'V015', 'V017', 'V019'], 'VEH005': []}


### Validacion

In [122]:
rutas = rutas_generadas
verbose = False

for veh, ruta in rutas.items():
    print(f"Vehículo {veh}: {ruta}")
    if ruta == []:
        print(f"Vehículo {veh} no tiene viajes asignados.")
        continue
    
    vehiculo = data_vehiculos[data_vehiculos['id'] == veh].iloc[0]
    # === Checkeo de factiblidad de las rutas asignadas ===
    
    # Criterio 1: que cada viaje asignado a un vehículo cumpla con las restricciones de capacidad
    for idx, viaje_id in enumerate(ruta):
        viaje = data_viajes[data_viajes['id'] == viaje_id].iloc[0]
        if vehiculo.capacidad < viaje.num_pasajeros:
            print(f"Error: Vehículo {veh} no tiene capacidad para el viaje {viaje_id}")
        else:
            if verbose: print(f"Vehículo {veh} tiene capacidad {vehiculo.capacidad} para el viaje {viaje_id} con {viaje.num_pasajeros} pasajeros")
    
    # Criterio 2: sucesión de viajes --> fin viaje i + deadhead i,j <= hora presentación viaje j
        viaje_siguiente = data_viajes[data_viajes['id'] == ruta[idx+1]].iloc[0] if idx + 1 < len(ruta) else None
        if viaje_siguiente is not None:
            dh = haversine(viaje.lat_destino, viaje.lon_destino, viaje_siguiente.lat_origen, viaje_siguiente.lon_origen)
            tiempo_llegada = dh / VELOCIDAD_MEDIA
            hora_llegada = viaje.hora_finalizacion + tiempo_llegada
            if not(hora_llegada <= viaje_siguiente.hora_presentacion):
                print(f"Error: Vehículo {veh} no llega a tiempo para el viaje {viaje_siguiente.id} después de completar el viaje {viaje_id}")
            else:
                if verbose: print(f"Vehículo {veh} llega a tiempo para el viaje {viaje_siguiente.id} después de completar el viaje {viaje_id}")
    
    # Criterio 3: primer viaje i - deadhead depot,i despues de inicio de ventana operativa
    primer_viaje = data_viajes[data_viajes['id'] == ruta[0]].iloc[0]
    dh = haversine(vehiculo.lat_inicio_jornada, vehiculo.lon_inicio_jornada, primer_viaje.lat_origen, primer_viaje.lon_origen)
    tiempo_llegada = dh / VELOCIDAD_MEDIA   

    hora_llegada = vehiculo.hora_inicio_jornada + tiempo_llegada
    if not(hora_llegada <= primer_viaje.hora_presentacion):
        print(f"Error: Vehículo {veh} no llega a tiempo para el primer viaje {primer_viaje.id} después de salir del depósito: {vehiculo.hora_inicio_jornada} + {tiempo_llegada} > {primer_viaje.hora_presentacion}")
    else:
        if verbose: print(f"Vehículo {veh} llega a tiempo para el primer viaje {primer_viaje.id} después de salir del depósito")
    # Criterio 4: fin viaje i + deadhead i, depot antes de fin de ventana operativa
    ultimo_viaje = data_viajes[data_viajes['id'] == ruta[-1]].iloc[0]
    dh = haversine(ultimo_viaje.lat_destino, ultimo_viaje.lon_destino, vehiculo.lat_fin_jornada, vehiculo.lon_fin_jornada)
    tiempo_llegada = dh / VELOCIDAD_MEDIA   
    hora_llegada = ultimo_viaje.hora_finalizacion + tiempo_llegada
    if not(hora_llegada <= vehiculo.hora_fin_jornada):
        print(f"Error: Vehículo {veh} no llega a tiempo al depósito después de completar el último viaje {ultimo_viaje.id}: {ultimo_viaje.hora_finalizacion} + {tiempo_llegada} > {vehiculo.hora_fin_jornada}")
    else:
        if verbose: print(f"Vehículo {veh} llega a tiempo para el ultimo viaje {ultimo_viaje.id} después de salir del depósito")

Vehículo VEH001: ['V001', 'V003', 'V006', 'V008', 'V010', 'V013', 'V020', 'V022', 'V023', 'V025', 'V027', 'V029']
Vehículo VEH002: []
Vehículo VEH002 no tiene viajes asignados.
Vehículo VEH003: ['V014', 'V016', 'V018', 'V021', 'V024', 'V026', 'V028', 'V030']
Vehículo VEH004: ['V002', 'V004', 'V005', 'V007', 'V009', 'V011', 'V012', 'V015', 'V017', 'V019']
Vehículo VEH005: []
Vehículo VEH005 no tiene viajes asignados.


In [ ]:
def validacion_de_rutas(rutas: dict, data_viajes: dict, data_vehiculos:dict):
    for ruta in rutas.values():
        
